Load the embeddings

In [3]:
import numpy as np
import pandas as pd

notes = pd.read_csv(
    r'...\NOTEEVENTS.csv.gz',
    dtype={4: str, 5: str}  # or int, float, etc. depending on data
)

notes = notes[notes["CATEGORY"].isin(["Discharge summary"])]
notes = notes.dropna(subset=["TEXT", "HADM_ID"])

patient_texts = notes.groupby("HADM_ID")["TEXT"].apply(lambda x: "\n".join(x)).reset_index()

In [5]:
# Ensure consistent ID type
patient_texts['HADM_ID'] = patient_texts['HADM_ID'].astype(str)

# Load clinical note embeddings
loaded = np.load(r'...\useremb.npz')
clinical_embs = loaded['array1']  # shape: (n_users, emb_dim)

projected_note_embs = []

for i, subj_id in enumerate(patient_texts['HADM_ID']):
    clinical_emb = clinical_embs[i]
    
    projected_note_embs.append(clinical_emb)

projected_note_embs = np.array(projected_note_embs)
print(f"Final note embedding shape: {projected_note_embs.shape}")

Final note embedding shape: (52726, 768)


In [6]:
import torch
import torch.nn as nn
import numpy as np

class EmbeddingProjector(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(input_dim, output_dim),
            nn.ReLU(),  # Optional: or GELU, Tanh
            nn.Dropout(0.1)
        )

    def forward(self, x):
        return self.proj(x)


In [7]:
# Convert to tensor
combined_tensor = torch.tensor(projected_note_embs, dtype=torch.float32)

# Project to original dimension (or any size)
input_dim = combined_tensor.shape[1]
output_dim = 128 
projector = EmbeddingProjector(input_dim, output_dim)

# Forward pass (no training yet)
with torch.no_grad():
    projected_tensor = projector(combined_tensor)

# Convert back to NumPy if needed for Cornac
projected_note_embs = projected_tensor.numpy()


In [8]:
projected_note_embs.shape

(52726, 128)

In [8]:
drug_embeddings = np.load(r'...\drug_embeddings.npy', allow_pickle=True)

In [9]:
import torch
# Convert to tensor
combined_tensor = torch.tensor(drug_embeddings, dtype=torch.float32)

# Project to original dimension (or any size)
input_dim = combined_tensor.shape[1]
output_dim = 128
projector = EmbeddingProjector(input_dim, output_dim)

# Forward pass (no training yet)
with torch.no_grad():
    projected_tensor = projector(combined_tensor)

# Convert back to NumPy if needed for Cornac
projected_drug_embeddings = projected_tensor.numpy()

In [10]:
projected_drug_embeddings.shape

(499, 128)

Safety Metrics

In [11]:
from cornac.metrics import RankingMetric

class DDIRate(RankingMetric):
    def __init__(self, ddi_matrix, k=10, name="DDI@10"):
        """
        Parameters:
        - ddi_pairs: set of (drug_id_1, drug_id_2) tuples indicating known DDIs.
        - k: number of top predicted items to consider per user.
        """
        super().__init__(name=name, k=k)
        self.ddi_matrix = ddi_matrix

    def compute(self, gt_pos, gt_neg, pd_rank, pd_scores, item_indices=None):
        top_k_items = pd_rank[:self.k]
        ddi_count = 0
        total_pairs = 0

        for i in range(len(top_k_items)):
            for j in range(i + 1, len(top_k_items)):
                d1, d2 = top_k_items[i], top_k_items[j]    
                if frozenset({d1, d2}) in self.ddi_matrix or frozenset({d2, d1}) in self.ddi_matrix:
                    ddi_count += 1
                total_pairs += 1

        ddi_rate = ddi_count / total_pairs if total_pairs > 0 else 0.0
        return ddi_rate


In [12]:
import numpy as np
from itertools import combinations

class ToxicityDDIRate(RankingMetric):
    def __init__(self, toxicity_matrix, k=10, name="ToxicityDDI@10"):
        """
        Parameters:
        - toxicity_matrix: 2D NumPy array or sparse matrix where toxicity_matrix[i, j] 
                           gives the toxicity score of the DDI between drugs i and j.
                           (0 if no interaction, >0 if interaction exists)
        - k: number of top predicted items to consider per user.
        """
        super().__init__(name=name, k=k)
        self.toxicity_matrix = toxicity_matrix

    def compute(self, gt_pos, gt_neg, pd_rank, pd_scores, item_indices=None):
        top_k_items = pd_rank[:self.k]
        if len(top_k_items) < 2:
            return 0.0

        # All unordered pairs among top-k
        pairs = np.array(list(combinations(top_k_items, 2)))

        # Sum toxicity of interactions among top-k items
        toxicity_sum = self.toxicity_matrix[pairs[:, 0], pairs[:, 1]].sum()
        total_pairs = len(pairs)

        return toxicity_sum / total_pairs if total_pairs > 0 else 0.0


In [15]:
import pickle
with open(r'...\mapped_ddi_pairs.pkl', 'rb') as f:
    mapped_ddi_pairs = pickle.load(f)

In [16]:
print(mapped_ddi_pairs[0])

('clomipramine', 'itraconazole', 'minor')


ToxicFreeMed 

In [17]:
import numpy as np
import copy
from tqdm.auto import trange
from cornac.models import Recommender
from cornac.utils.init_utils import uniform, zeros


def normalize_rows(mat):
    """Row-wise L2 normalization."""
    norms = np.linalg.norm(mat, axis=1, keepdims=True)
    norms[norms == 0] = 1
    return mat / norms


class ToxicFreeMed(Recommender):
    """
    MultiTask BPR with:
    - Pretrained patient/drug embeddings
    - Optional DDI toxicity multitask loss (vectorized)
    """

    def __init__(
        self,
        # pretrained
        patient_embeddings=None,
        drug_embeddings=None,
        fold_uid_map=None,
        fold_iid_map=None,
        residual_scale=0.01,
        # multitask BPR
        k=50,
        max_iter=100,
        learning_rate=0.01,
        lambda_reg=0.001,
        alpha=0.8,
        ddi_pairs=None,
        # misc
        verbose=False,
        seed=None,
    ):
        super().__init__(name="ToxicFreeMed", trainable=True, verbose=verbose)

        # embedding dim (override if pretrained provided)
        self.k = (
            patient_embeddings.shape[1]
            if patient_embeddings is not None
            else (drug_embeddings.shape[1] if drug_embeddings is not None else k)
        )

        # store pretrained + maps
        self.patient_embeddings = (
            normalize_rows(patient_embeddings) if patient_embeddings is not None else None
        )
        self.drug_embeddings = (
            normalize_rows(drug_embeddings) if drug_embeddings is not None else None
        )
        self.fold_uid_map = fold_uid_map or {}
        self.fold_iid_map = fold_iid_map or {}

        self.residual_scale = residual_scale

        self.max_iter = max_iter
        self.learning_rate = learning_rate
        self.lambda_reg = lambda_reg
        self.alpha = alpha
        self.ddi_pairs = ddi_pairs if ddi_pairs is not None else []
        self.seed = seed
        self.rng = np.random.RandomState(seed)


        # save params for clone
        self._init_params = copy.deepcopy(locals())
        self._init_params.pop("self")

    def _init_factors(self, train_set):
        # ---- users ----
        self.u_factors = (uniform((train_set.num_users, self.k),
                                  random_state=self.rng,
                                  dtype=np.float32) - 0.5) / self.k
        covered_users = 0
        if self.patient_embeddings is not None and self.fold_uid_map:
            for uid in range(train_set.num_users):
                raw_uid = train_set.user_ids[uid]
                if raw_uid in self.fold_uid_map:
                    emb_idx = self.fold_uid_map[raw_uid]
                    base = self.patient_embeddings[emb_idx]
                    if not np.allclose(base, 0):
                        covered_users += 1
                    self.u_factors[uid] = base + self.residual_scale * (
                        uniform((1, self.k), random_state=self.rng, dtype=np.float32) - 0.5
                    ) / self.k
        if self.verbose:
            print(f"Users with pretrained emb: {covered_users}/{train_set.num_users}")

        # ---- items ----
        self.i_factors = (uniform((train_set.num_items, self.k),
                                  random_state=self.rng,
                                  dtype=np.float32) - 0.5) / self.k
        covered_items = 0
        if self.drug_embeddings is not None and self.fold_iid_map:
            for iid in range(train_set.num_items):
                raw_iid = train_set.item_ids[iid].lower().strip()
                if raw_iid in self.fold_iid_map:
                    emb_idx = self.fold_iid_map[raw_iid]
                    base = self.drug_embeddings[emb_idx]
                    if not np.allclose(base, 0):
                        covered_items += 1
                    self.i_factors[iid] = base + self.residual_scale * (
                        uniform((1, self.k), random_state=self.rng, dtype=np.float32) - 0.5
                    ) / self.k
        if self.verbose:
            print(f"Items with pretrained emb: {covered_items}/{train_set.num_items}")

        # biases
        self.i_biases = zeros(train_set.num_items, dtype=np.float32)

    def _prepare_data(self, train_set):
        X = train_set.matrix
        user_counts = np.ediff1d(X.indptr)
        user_ids = np.repeat(np.arange(train_set.num_users), user_counts)
        return X, user_counts, user_ids

    def fit(self, train_set, val_set=None):
        super().fit(train_set, val_set)
        self._init_factors(train_set)
        X, _, user_ids = self._prepare_data(train_set)
        neg_item_ids = np.arange(train_set.num_items, dtype=np.int32)

        with trange(self.max_iter, disable=not self.verbose) as progress:
            for epoch in progress:
                correct, skipped = self._fit_sgd(user_ids, X.indices, X.indptr, neg_item_ids)
                if self.verbose:
                    progress.set_postfix({
                        "correct": "%.2f%%" % (100.0 * correct / (len(user_ids) - skipped)),
                        "skipped": "%.2f%%" % (100.0 * skipped / len(user_ids))
                    })
        return self



    def _fit_sgd(self, user_ids, item_ids, indptr, neg_item_ids):
        item_id_set = set(item_ids)

        # filter DDI pairs
        if self.ddi_pairs:
            filtered = [(d1, d2, tox) for d1, d2, tox in self.ddi_pairs
                        if d1 in item_id_set and d2 in item_id_set]
            if filtered:
                d1_idx, d2_idx, tox_vals = zip(*filtered)
                d1_idx, d2_idx, tox_vals = map(np.array, (d1_idx, d2_idx, tox_vals))
            else:
                d1_idx = d2_idx = tox_vals = np.array([])
        else:
            d1_idx = d2_idx = tox_vals = np.array([])

        # neg items per user
        user_neg_sets = {}
        for u in np.unique(user_ids):
            start, end = indptr[u], indptr[u + 1]
            user_neg_sets[u] = np.array(list(set(neg_item_ids) - set(item_ids[start:end])))

        # pos/neg triplets
        u_array, i_array, j_array = [], [], []
        for u in np.unique(user_ids):
            pos_items = item_ids[indptr[u]:indptr[u + 1]]
            neg_items = user_neg_sets[u]
            n_samples = len(pos_items)
            if n_samples == 0:
                continue
            u_array.extend([u] * n_samples)
            i_array.extend(pos_items)
            j_array.extend(self.rng.choice(neg_items, size=n_samples, replace=True))
        u_array, i_array, j_array = map(np.array, (u_array, i_array, j_array))


        # --- BPR forward ---
        x_ui = self.i_biases[i_array] + np.sum(self.u_factors[u_array] * self.i_factors[i_array], axis=1)
        x_uj = self.i_biases[j_array] + np.sum(self.u_factors[u_array] * self.i_factors[j_array], axis=1)
        x_uij = x_ui - x_uj
        z = 1.0 / (1.0 + np.exp(x_uij))
        correct = np.sum(z < 0.5)
        skipped = 0        
        
        # --- BPR updates ---
        grad_u = z[:, None] * (self.i_factors[i_array] - self.i_factors[j_array]) - self.lambda_reg * self.u_factors[u_array]
        grad_i = z[:, None] * self.u_factors[u_array] - self.lambda_reg * self.i_factors[i_array]
        grad_j = -z[:, None] * self.u_factors[u_array] - self.lambda_reg * self.i_factors[j_array]

        np.add.at(self.u_factors, u_array, self.learning_rate * self.alpha * grad_u)
        np.add.at(self.i_factors, i_array, self.learning_rate * self.alpha * grad_i)
        np.add.at(self.i_factors, j_array, self.learning_rate * self.alpha * grad_j)
        np.add.at(self.i_biases, i_array, self.learning_rate * self.alpha * (z - self.lambda_reg * self.i_biases[i_array]))
        np.add.at(self.i_biases, j_array, self.learning_rate * self.alpha * (-z - self.lambda_reg * self.i_biases[j_array]))

        # --- DDI toxicity updates ---
        if d1_idx.size > 0:
            prod = np.sum(self.i_factors[d1_idx] * self.i_factors[d2_idx], axis=1) - tox_vals
            grad_d1 = prod[:, None] * self.i_factors[d2_idx]
            grad_d2 = prod[:, None] * self.i_factors[d1_idx]
            self.i_factors[d1_idx] -= self.learning_rate * (1 - self.alpha) * grad_d1
            self.i_factors[d2_idx] -= self.learning_rate * (1 - self.alpha) * grad_d2

        return correct, skipped

    def score(self, user_idx, item_idx=None):
        if item_idx is None:
            scores = np.copy(self.i_biases)
            scores += self.u_factors[user_idx] @ self.i_factors.T
            return scores
        else:
            return self.i_biases[item_idx] + self.u_factors[user_idx] @ self.i_factors[item_idx]

    def _get_init_params(self):
        return copy.deepcopy(self._init_params)

    def clone(self, new_params=None):
        import inspect
        params = copy.deepcopy(self._init_params)
        if new_params:
            params.update(new_params)
        init_params = inspect.signature(self.__init__).parameters
        filtered_params = {k: v for k, v in params.items() if k in init_params}
        return self.__class__(**filtered_params)


In [ ]:
# ------------------- EXTERNAL VALIDATION PIPELINE WITH STANDARD DEVIATIONS -------------------
# Train on MIMIC-III, Evaluate on eICU (Mapped to MIMIC Item Space)

# ------------------- Step 0: Imports -------------------
import numpy as np
import pandas as pd
import re
from tqdm import tqdm
from rapidfuzz import process
import cornac
from cornac.data import Dataset
from cornac.eval_methods import RatioSplit
from cornac.metrics import Recall, NDCG
import matplotlib.pyplot as plt
import warnings
from itertools import combinations
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.random_projection import GaussianRandomProjection
import json
import os
import torch
import torch.nn as nn
warnings.filterwarnings('ignore')


# ------------------- Step 1: Load Data -------------------
print("Loading datasets...")
mimic_anemia_df = pd.read_csv(r"..............user_drug_rating_visit_anemia.csv")
eicu_anemia_df = pd.read_csv(r"................user_drug_rating_visit_eicu_anemia.csv")
matched_df = pd.read_csv(r"............drugbank_mimic_rxcui_map.csv")

# ensure IDs are strings
if 'patient_texts' in dir():
    patient_texts['HADM_ID'] = patient_texts['HADM_ID'].astype(float).astype(int).astype(str)

print(f"\nDataset sizes:")
print(f"  MIMIC: {len(mimic_anemia_df)} interactions, {mimic_anemia_df['user'].nunique()} users, {mimic_anemia_df['item'].nunique()} drugs")
print(f"  eICU: {len(eicu_anemia_df)} interactions, {eicu_anemia_df['user'].nunique()} users, {eicu_anemia_df['item'].nunique()} drugs")

# ------------------- Step 2: Clean Drug Names -------------------
def clean_drug_name(name):
    if pd.isnull(name):
        return ""
    name = name.lower().strip()
    name = re.sub(
        r"\b\d+(\.\d+)?\s*(mg|ml|mcg|units|tablet|tab|capsule|cap|drop|syrup|patch|ointment|cream|injection|solution|suspension|oral|inj|dose|suppository)\b",
        "", name
    )
    name = re.sub(r"[^\w\s]", "", name)
    name = re.sub(r"\s+", " ", name)
    return name.strip()

def preprocess_dataset(df, dataset_name):
    print(f"\nPreprocessing {dataset_name} dataset...")
    df = df.dropna(subset=["user", "item", "rating"]).copy()
    df["user"] = df["user"].astype(str)
    df["item"] = df["item"].astype(str)
    df["rating"] = df["rating"].astype(float)
    df["clean_item"] = df["item"].apply(clean_drug_name)
    print(f"  {dataset_name}: {len(df)} ratings, {df['user'].nunique()} users, {df['item'].nunique()} drugs")
    return df

mimic_anemia_df = preprocess_dataset(mimic_anemia_df, "MIMIC-III Anemia")
eicu_anemia_df = preprocess_dataset(eicu_anemia_df, "eICU Anemia")
matched_df["clean_generic"] = matched_df["Generic_Name"].astype(str).apply(clean_drug_name)

# ------------------- Step 3: Fuzzy Match MIMIC Drugs to DrugBank -------------------
print("\n" + "="*80)
print("STEP 3: Fuzzy Match MIMIC Drugs to DrugBank")
print("="*80)

mimic_items = mimic_anemia_df["clean_item"].unique()
mimic_generics = matched_df["clean_generic"].unique().tolist()

mimic_lookup = {}
mimic_match_count = 0
for item in tqdm(mimic_items, desc="Matching MIMIC drugs"):
    match = process.extractOne(item, mimic_generics, score_cutoff=80)
    if match:
        mimic_lookup[item] = match[0]
        mimic_match_count += 1

mimic_anemia_df["matched_generic"] = mimic_anemia_df["clean_item"].map(mimic_lookup)
mimic_anemia_df["matched_generic"] = mimic_anemia_df["matched_generic"].fillna(mimic_anemia_df["clean_item"])
print(f"  Matched {mimic_match_count}/{len(mimic_items)} ({100*mimic_match_count/len(mimic_items):.1f}%)")

# ------------------- Step 4: Fuzzy Match eICU Drugs to DrugBank -------------------
print("\n" + "="*80)
print("STEP 4: Fuzzy Match eICU Drugs to DrugBank")
print("="*80)

eicu_items = eicu_anemia_df["clean_item"].unique()
eicu_lookup = {}
eicu_match_count = 0
for item in tqdm(eicu_items, desc="Matching eICU drugs"):
    match = process.extractOne(item, mimic_generics, score_cutoff=80)
    if match:
        eicu_lookup[item] = match[0]
        eicu_match_count += 1

eicu_anemia_df["matched_generic"] = eicu_anemia_df["clean_item"].map(eicu_lookup)
eicu_anemia_df["matched_generic"] = eicu_anemia_df["matched_generic"].fillna(eicu_anemia_df["clean_item"])
print(f"  Matched {eicu_match_count}/{len(eicu_items)} ({100*eicu_match_count/len(eicu_items):.1f}%)")

# ------------------- Step 5: Filter Users With Embeddings (MIMIC only) -------------------
if 'projected_note_embs' in dir() and 'patient_texts' in dir():
    subj_id_to_emb_idx = {sid: idx for idx, sid in enumerate(patient_texts["HADM_ID"])}
    mimic_anemia_df = mimic_anemia_df[mimic_anemia_df["user"].isin(subj_id_to_emb_idx)]

print(f"\nAfter user filtering (MIMIC only):")
print(f"  MIMIC: {len(mimic_anemia_df)} ratings, {mimic_anemia_df['user'].nunique()} users")
print(f"  eICU: {len(eicu_anemia_df)} ratings, {eicu_anemia_df['user'].nunique()} users")

# ------------------- Step 6: Prepare UIR for MIMIC -------------------
def prepare_uir_data(ratings_df, dataset_name):
    uir_data = list(zip(ratings_df["user"], ratings_df["matched_generic"], ratings_df["rating"]))
    cornac_data = Dataset.from_uir(uir_data, seed=123)
    print(f"  {dataset_name}: {len(uir_data)} interactions, {len(cornac_data.uid_map)} users, {len(cornac_data.iid_map)} items")
    return uir_data, cornac_data.uid_map, cornac_data.iid_map

mimic_uir, mimic_uid_map, mimic_iid_map = prepare_uir_data(mimic_anemia_df, "MIMIC-III")
print(f"\nMIMIC: {len(mimic_uir)} interactions, {len(mimic_uid_map)} users, {len(mimic_iid_map)} items")

# ------------------- Step 7: User/Item Embeddings (MIMIC only) -------------------
def align_user_embeddings(cornac_uid_map, raw_to_emb_idx, embeddings):
    mat = np.zeros((len(cornac_uid_map), embeddings.shape[1]))
    for raw_uid, internal_uid in cornac_uid_map.items():
        if raw_uid in raw_to_emb_idx:
            mat[internal_uid] = embeddings[raw_to_emb_idx[raw_uid]]
    return mat

def align_item_embeddings(cornac_iid_map, raw_to_emb_idx, embeddings):
    mat = np.zeros((len(cornac_iid_map), embeddings.shape[1]))
    for raw_iid, internal_iid in cornac_iid_map.items():
        key = raw_iid.lower().strip()
        if key in raw_to_emb_idx:
            mat[internal_iid] = embeddings[raw_to_emb_idx[key]]
    return mat

if 'projected_drug_embeddings' in dir():
    drug_id_to_index = {row["clean_generic"]: idx for idx, row in matched_df.iterrows()}
    mimic_item_emb = align_item_embeddings(mimic_iid_map, drug_id_to_index, projected_drug_embeddings)
else:
    mimic_item_emb = None

if 'projected_note_embs' in dir() and 'patient_texts' in dir():
    mimic_user_emb = align_user_embeddings(mimic_uid_map, subj_id_to_emb_idx, projected_note_embs)
else:
    mimic_user_emb = None

print(f"\nMIMIC Embeddings aligned:")
print(f"  User embeddings: {mimic_user_emb.shape if mimic_user_emb is not None else 'None'}")
print(f"  Item embeddings: {mimic_item_emb.shape if mimic_item_emb is not None else 'None'}")

# ------------------- Step 8: Load and Project eICU Patient Embeddings -------------------
print("\n" + "="*80)
print("STEP 8: Loading and Projecting eICU Patient Embeddings")
print("="*80)

embedding_dim = 128
eicu_emb_path = r'................eicu_patient_embeddings.npy'
eicu_map_path = r'.................eicu_patient_id_to_idx.json'

if os.path.exists(eicu_emb_path) and os.path.exists(eicu_map_path):
    eicu_patient_emb = np.load(eicu_emb_path)
    with open(eicu_map_path, 'r') as f:
        eicu_patient_id_to_idx = json.load(f)
    print(f"  Loaded eICU patient embeddings: {eicu_patient_emb.shape}")
    
    # Project from 768 to 128 dimensions
    if eicu_patient_emb.shape[1] != embedding_dim:
        print(f"  Projecting from {eicu_patient_emb.shape[1]} to {embedding_dim} dimensions...")
        
        # Method 1: Using EmbeddingProjector (PyTorch)
        projector = EmbeddingProjector(
            input_dim=eicu_patient_emb.shape[1], 
            output_dim=embedding_dim
        )
        projector.eval()
        
        with torch.no_grad():
            eicu_tensor = torch.tensor(eicu_patient_emb, dtype=torch.float32)
            eicu_projected = projector(eicu_tensor).numpy()
        
        # Normalize
        norms = np.linalg.norm(eicu_projected, axis=1, keepdims=True)
        norms[norms == 0] = 1
        eicu_patient_emb = eicu_projected / norms
        
        print(f"  Projected embeddings shape: {eicu_patient_emb.shape}")
else:
    print("  No eICU embeddings found. Creating random embeddings.")
    eicu_patient_emb = np.random.randn(len(eicu_anemia_df['user'].unique()), embedding_dim)
    norms = np.linalg.norm(eicu_patient_emb, axis=1, keepdims=True)
    norms[norms == 0] = 1
    eicu_patient_emb = eicu_patient_emb / norms
    eicu_patient_id_to_idx = {}

print(f"  Final eICU embeddings shape: {eicu_patient_emb.shape}")

# ------------------- Step 9: Create DDI Structures (MIMIC only) -------------------
print("\n" + "="*80)
print("STEP 9: Creating DDI Structures")
print("="*80)

toxicity_map = {"minor": 1.0, "moderate": 2.0, "major": 3.0}

mimic_current_drugs = set(mimic_iid_map.keys())
mimic_filtered_ddi_pairs = []
mimic_ddi_index_pairs = []

for d1, d2, tox in mapped_ddi_pairs:
    d1_clean = d1.lower().strip()
    d2_clean = d2.lower().strip()
    if d1_clean in mimic_current_drugs and d2_clean in mimic_current_drugs:
        mimic_filtered_ddi_pairs.append((d1_clean, d2_clean, tox))
        if d1_clean in mimic_iid_map and d2_clean in mimic_iid_map:
            mimic_ddi_index_pairs.append((mimic_iid_map[d1_clean], mimic_iid_map[d2_clean], toxicity_map.get(tox, 1.0)))

mimic_tox_matrix = np.zeros((len(mimic_iid_map), len(mimic_iid_map)))
mimic_ddi_set = set()

for d1, d2, tox in mimic_filtered_ddi_pairs:
    if d1 in mimic_iid_map and d2 in mimic_iid_map:
        i, j = mimic_iid_map[d1], mimic_iid_map[d2]
        mimic_tox_matrix[i, j] = mimic_tox_matrix[j, i] = toxicity_map.get(tox, 1.0)
        mimic_ddi_set.add(frozenset([i, j]))

print(f"  MIMIC DDI pairs: {len(mimic_filtered_ddi_pairs)}")

# ============================================================================
# HELPER FUNCTION: Evaluate recommendations with all metrics
# ============================================================================

def evaluate_recommendations(ranked_items, true_items, ddi_set, tox_matrix, k=10):
    true_set = set(true_items)
    ranked_k = ranked_items[:k]
    
    recall = len(set(ranked_k) & true_set) / min(len(true_set), k)
    
    dcg = 0
    for rank, item in enumerate(ranked_k):
        if item in true_set:
            dcg += 1 / np.log2(rank + 2)
    idcg = sum(1 / np.log2(i + 2) for i in range(min(len(true_set), k)))
    ndcg = dcg / idcg if idcg > 0 else 0
    
    if len(ranked_k) >= 2:
        ddi_count = 0
        for i in range(len(ranked_k)):
            for j in range(i + 1, len(ranked_k)):
                if frozenset([ranked_k[i], ranked_k[j]]) in ddi_set:
                    ddi_count += 1
        total_pairs = len(ranked_k) * (len(ranked_k) - 1) / 2
        ddi_rate = ddi_count / total_pairs if total_pairs > 0 else 0
    else:
        ddi_rate = 0.0
    
    if len(ranked_k) >= 2:
        toxicity_sum = 0
        for i in range(len(ranked_k)):
            for j in range(i + 1, len(ranked_k)):
                toxicity_sum += tox_matrix[ranked_k[i], ranked_k[j]]
        total_pairs = len(ranked_k) * (len(ranked_k) - 1) / 2
        toxicity_ddi_rate = toxicity_sum / total_pairs if total_pairs > 0 else 0
    else:
        toxicity_ddi_rate = 0.0
    
    return recall, ndcg, ddi_rate, toxicity_ddi_rate

# ============================================================================
# TRAINING AND EVALUATION WITH MULTIPLE RUNS FOR STD DEV
# ============================================================================

ALPHA_VALUE = 0.005
N_RUNS = 5  # Number of runs for standard deviation

print("\n" + "="*80)
print(f"EXTERNAL VALIDATION: α = {ALPHA_VALUE}")
print(f"Number of runs for std dev: {N_RUNS}")
print("="*80)

# Store results from multiple runs
mimic_recall_runs = []
mimic_ndcg_runs = []
mimic_ddi_runs = []
mimic_tox_runs = []

eicu_recall_runs = []
eicu_ndcg_runs = []
eicu_ddi_runs = []
eicu_tox_runs = []

# Pre-compute eICU mappings (same across runs)
eicu_uir_original = list(zip(eicu_anemia_df["user"], eicu_anemia_df["matched_generic"], eicu_anemia_df["rating"]))

# Map eICU drugs to MIMIC item indices
eicu_drug_to_mimic_idx = {}
for eicu_drug in set([d for _, d, _ in eicu_uir_original]):
    if eicu_drug in mimic_iid_map:
        eicu_drug_to_mimic_idx[eicu_drug] = mimic_iid_map[eicu_drug]
    else:
        match = process.extractOne(eicu_drug, list(mimic_iid_map.keys()), score_cutoff=80)
        if match:
            eicu_drug_to_mimic_idx[eicu_drug] = mimic_iid_map[match[0]]

eicu_uir_mapped = []
for user, drug, rating in eicu_uir_original:
    if drug in eicu_drug_to_mimic_idx:
        eicu_uir_mapped.append((user, eicu_drug_to_mimic_idx[drug], rating))

eicu_mapped_dataset = Dataset.from_uir(eicu_uir_mapped, seed=123)
eicu_mapped_uid_map = eicu_mapped_dataset.uid_map

# Build eICU user items
eicu_all_user_items = {}
for user, item_idx, rating in eicu_uir_mapped:
    if user not in eicu_all_user_items:
        eicu_all_user_items[user] = []
    eicu_all_user_items[user].append(item_idx)

eicu_user_to_idx = {user: idx for idx, user in enumerate(eicu_mapped_uid_map.keys())}

for run in range(N_RUNS):
    print(f"\n{'='*60}")
    print(f"RUN {run+1}/{N_RUNS}")
    print(f"{'='*60}")
    
    # Set different seed for each run
    run_seed = 42 + run
    
    # ------------------- Create MIMIC Split -------------------
    mimic_split = RatioSplit(
        data=mimic_uir,
        test_size=0.2,
        exclude_unknowns=True,
        rating_threshold=1.0,
        verbose=False,
        seed=run_seed,
    )
    
    # ------------------- Train Model -------------------
    model = ToxicFreeMed(
        k=128,
        alpha=ALPHA_VALUE,
        max_iter=1000,
        learning_rate=0.001,
        ddi_pairs=mimic_ddi_index_pairs,
        lambda_reg=0.001,
        verbose=False,
        seed=run_seed,
        patient_embeddings=mimic_user_emb,
        drug_embeddings=mimic_item_emb,
        fold_uid_map=mimic_uid_map,
        fold_iid_map=mimic_iid_map,
        train_user_embeddings=True,
        train_item_embeddings=True,
        residual_scale=0.01,
        auto_lr=False
    )
    
    print(f"\nTraining on 80% of MIMIC data (Run {run+1})...")
    model.fit(mimic_split.train_set)
    
    # ------------------- Evaluate MIMIC Test Set -------------------
    test_matrix = mimic_split.test_set.matrix
    test_users, test_items = test_matrix.nonzero()
    test_ratings = test_matrix.data
    
    mimic_test_user_items = {}
    for u, i, r in zip(test_users, test_items, test_ratings):
        if u not in mimic_test_user_items:
            mimic_test_user_items[u] = []
        mimic_test_user_items[u].append(i)
    
    mimic_all_items = list(range(len(mimic_iid_map)))
    
    mimic_recalls = []
    mimic_ndcgs = []
    mimic_ddi_rates = []
    mimic_toxicity_ddi = []
    
    for u, true_items in mimic_test_user_items.items():
        scores = []
        for i in mimic_all_items:
            try:
                score = model.score(u, i)
            except:
                if hasattr(model, 'i_biases') and i < len(model.i_biases):
                    score = model.i_biases[i]
                else:
                    score = 0.0
            scores.append((i, score))
        
        scores.sort(key=lambda x: x[1], reverse=True)
        ranked = [x[0] for x in scores[:10]]
        
        recall, ndcg, ddi_rate, toxicity_ddi = evaluate_recommendations(
            ranked, true_items, mimic_ddi_set, mimic_tox_matrix, k=10
        )
        mimic_recalls.append(recall)
        mimic_ndcgs.append(ndcg)
        mimic_ddi_rates.append(ddi_rate)
        mimic_toxicity_ddi.append(toxicity_ddi)
    
    mimic_recall_runs.append(np.mean(mimic_recalls))
    mimic_ndcg_runs.append(np.mean(mimic_ndcgs))
    mimic_ddi_runs.append(np.mean(mimic_ddi_rates))
    mimic_tox_runs.append(np.mean(mimic_toxicity_ddi))
    
    print(f"  MIMIC - Recall@10: {mimic_recall_runs[-1]:.4f}, NDCG@10: {mimic_ndcg_runs[-1]:.4f}")
    
    # ------------------- Create eICU User Embedding Matrix -------------------
    eicu_user_emb_eval = np.zeros((len(eicu_mapped_uid_map), embedding_dim))
    for raw_user, internal_idx in eicu_mapped_uid_map.items():
        if raw_user in eicu_patient_id_to_idx:
            emb_idx = eicu_patient_id_to_idx[raw_user]
            eicu_user_emb_eval[internal_idx] = eicu_patient_emb[emb_idx]
        else:
            random_emb = (np.random.rand(embedding_dim) - 0.5) / embedding_dim
            norm = np.linalg.norm(random_emb)
            if norm > 0:
                random_emb = random_emb / norm
            eicu_user_emb_eval[internal_idx] = random_emb
    
    # Save original user factors and replace with eICU embeddings
    original_u_factors = model.u_factors.copy()
    model.u_factors = eicu_user_emb_eval
    
    all_items = list(range(len(mimic_iid_map)))
    
    eicu_recalls = []
    eicu_ndcgs = []
    eicu_ddi_rates = []
    eicu_toxicity_ddi = []
    
    for user, true_items in eicu_all_user_items.items():
        if user not in eicu_user_to_idx:
            continue
        
        user_idx = eicu_user_to_idx[user]
        
        scores = []
        for item_idx in all_items:
            try:
                score = model.score(user_idx, item_idx)
            except:
                score = 0.0
            scores.append((item_idx, score))
        
        scores.sort(key=lambda x: x[1], reverse=True)
        ranked = [x[0] for x in scores[:10]]
        
        recall, ndcg, ddi_rate, toxicity_ddi = evaluate_recommendations(
            ranked, true_items, mimic_ddi_set, mimic_tox_matrix, k=10
        )
        eicu_recalls.append(recall)
        eicu_ndcgs.append(ndcg)
        eicu_ddi_rates.append(ddi_rate)
        eicu_tox_ddi.append(toxicity_ddi)
    
    # Restore original user factors
    model.u_factors = original_u_factors
    
    eicu_recall_runs.append(np.mean(eicu_recalls))
    eicu_ndcg_runs.append(np.mean(eicu_ndcgs))
    eicu_ddi_runs.append(np.mean(eicu_ddi_rates))
    eicu_tox_runs.append(np.mean(eicu_toxicity_ddi))
    
    print(f"  eICU - Recall@10: {eicu_recall_runs[-1]:.4f}, NDCG@10: {eicu_ndcg_runs[-1]:.4f}")

# ============================================================================
# RESULTS WITH STANDARD DEVIATIONS
# ============================================================================

print("\n" + "="*80)
print("FINAL RESULTS WITH STANDARD DEVIATIONS")
print("="*80)

mimic_results = {
    'recall_10': np.mean(mimic_recall_runs),
    'recall_10_std': np.std(mimic_recall_runs),
    'ndcg_10': np.mean(mimic_ndcg_runs),
    'ndcg_10_std': np.std(mimic_ndcg_runs),
    'ddi_10': np.mean(mimic_ddi_runs),
    'ddi_10_std': np.std(mimic_ddi_runs),
    'toxicity_ddi_10': np.mean(mimic_tox_runs),
    'toxicity_ddi_10_std': np.std(mimic_tox_runs)
}

eicu_results = {
    'recall_10': np.mean(eicu_recall_runs),
    'recall_10_std': np.std(eicu_recall_runs),
    'ndcg_10': np.mean(eicu_ndcg_runs),
    'ndcg_10_std': np.std(eicu_ndcg_runs),
    'ddi_10': np.mean(eicu_ddi_runs),
    'ddi_10_std': np.std(eicu_ddi_runs),
    'toxicity_ddi_10': np.mean(eicu_tox_runs),
    'toxicity_ddi_10_std': np.std(eicu_tox_runs)
}

print(f"\n{'Metric':<20} {'MIMIC (mean ± std)':<35} {'eICU (mean ± std)':<35}")
print("-" * 90)
print(f"{'Recall@10':<20} {mimic_results['recall_10']:.4f} ± {mimic_results['recall_10_std']:.4f}                    {eicu_results['recall_10']:.4f} ± {eicu_results['recall_10_std']:.4f}")
print(f"{'NDCG@10':<20} {mimic_results['ndcg_10']:.4f} ± {mimic_results['ndcg_10_std']:.4f}                    {eicu_results['ndcg_10']:.4f} ± {eicu_results['ndcg_10_std']:.4f}")
print(f"{'DDI@10':<20} {mimic_results['ddi_10']:.4f} ± {mimic_results['ddi_10_std']:.4f}                    {eicu_results['ddi_10']:.4f} ± {eicu_results['ddi_10_std']:.4f}")
print(f"{'Toxicity@10':<20} {mimic_results['toxicity_ddi_10']:.4f} ± {mimic_results['toxicity_ddi_10_std']:.4f}                    {eicu_results['toxicity_ddi_10']:.4f} ± {eicu_results['toxicity_ddi_10_std']:.4f}")

# Generalization metrics
if mimic_results['recall_10'] > 0:
    gen_recall = (eicu_results['recall_10'] / mimic_results['recall_10']) * 100
    gen_recall_std = gen_recall * np.sqrt((eicu_results['recall_10_std']/eicu_results['recall_10'])**2 + 
                                           (mimic_results['recall_10_std']/mimic_results['recall_10'])**2)
    
    gen_ndcg = (eicu_results['ndcg_10'] / mimic_results['ndcg_10']) * 100
    gen_ndcg_std = gen_ndcg * np.sqrt((eicu_results['ndcg_10_std']/eicu_results['ndcg_10'])**2 + 
                                       (mimic_results['ndcg_10_std']/mimic_results['ndcg_10'])**2)
    
    print(f"\n📊 GENERALIZATION (MIMIC → eICU):")
    print(f"  Recall@10: {gen_recall:.1f} ± {gen_recall_std:.1f}% of MIMIC performance")
    print(f"  NDCG@10: {gen_ndcg:.1f} ± {gen_ndcg_std:.1f}% of MIMIC performance")

print(f"\n  Shared item space: {len(mimic_iid_map)} MIMIC drugs")
print(f"  eICU drugs mapped to MIMIC: {len(eicu_mapped_dataset.iid_map)}")
print(f"  Number of runs: {N_RUNS}")

print("\n" + "="*80)
print("✅ EXTERNAL VALIDATION COMPLETE")
print(f"   Model trained on 80% of MIMIC data ({N_RUNS} runs with different seeds)")
print("   eICU evaluation uses SAME ITEM SPACE (MIMIC item indices)")
print("   eICU evaluation uses projected eICU patient embeddings (768 → 128)")
print("   Safety metrics computed on shared item space")
print("="*80)